In [561]:
import pandas as pd
import numpy as np

### 1. Data Loading and Initial Cleaning

In [562]:
df = pd.read_csv('SaaS-Sales.csv')

In [563]:
print("---DATASET SIZE---")
print(f"\nRows count: {df.shape[0]}")
print(f"Columns count: {df.shape[1]}")


---DATASET SIZE---

Rows count: 9994
Columns count: 19


In [564]:
print("---COLUMN NAMES---")
df.columns

---COLUMN NAMES---


Index(['Row ID', 'Order ID', 'Order Date', 'Date Key', 'Contact Name',
       'Country', 'City', 'Region', 'Subregion', 'Customer', 'Customer ID',
       'Industry', 'Segment', 'Product', 'License', 'Sales', 'Quantity',
       'Discount', 'Profit'],
      dtype='str')

In [565]:
print("---DATA TYPES OF COLUMNS---")
df.dtypes

---DATA TYPES OF COLUMNS---


Row ID            int64
Order ID            str
Order Date          str
Date Key          int64
Contact Name        str
Country             str
City                str
Region              str
Subregion           str
Customer            str
Customer ID       int64
Industry            str
Segment             str
Product             str
License             str
Sales           float64
Quantity          int64
Discount        float64
Profit          float64
dtype: object

In [566]:
print("---TIMESPAN OF ANALYSIS---\n")
df["Order Date"] = pd.to_datetime(df["Order Date"])
print(f"Timeline start: {df["Order Date"].min()}")
print(f"Timeline end: {df["Order Date"].max()}")

---TIMESPAN OF ANALYSIS---

Timeline start: 2020-01-04 00:00:00
Timeline end: 2023-12-31 00:00:00


In [567]:
print("---RENAMING THE COLUMNS---\n")
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.lower()
)

df = df.rename(columns={"row_id": "transaction_id"})

print(df.columns)



---RENAMING THE COLUMNS---

Index(['transaction_id', 'order_id', 'order_date', 'date_key', 'contact_name',
       'country', 'city', 'region', 'subregion', 'customer', 'customer_id',
       'industry', 'segment', 'product', 'license', 'sales', 'quantity',
       'discount', 'profit'],
      dtype='str')


In [568]:
print("---MISSING VALUES PER COLUMN---")
df.isnull().sum()

---MISSING VALUES PER COLUMN---


transaction_id    0
order_id          0
order_date        0
date_key          0
contact_name      0
country           0
city              0
region            0
subregion         0
customer          0
customer_id       0
industry          0
segment           0
product           0
license           0
sales             0
quantity          0
discount          0
profit            0
dtype: int64

In [569]:
print("---CHECKING FOR DUPLICATES---\n")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate id column: {df["transaction_id"].duplicated().sum()}")

---CHECKING FOR DUPLICATES---

Duplicate rows: 0
Duplicate id column: 0


### 2. Baseline Data Exploration

In [570]:
print("---OVERVIEW OF THE DATASET---")
df.sample(5)

---OVERVIEW OF THE DATASET---


,transaction_id,order_id,order_date,date_key,contact_name,country,city,region,subregion,customer,customer_id,industry,segment,product,license,sales,quantity,discount,profit
2659,2660,AMER-2021-135538,2021-12-24,20211224,Justin Carr,Chile,Puente Alto,AMER,LATAM,ConocoPhillips,1024,Energy,Strategic,FinanceHub,SF32D8JLCD,883.840,4,0.2,99.4320
1285,1286,EMEA-2022-119445,2022-06-27,20220627,Jonathan Brown,Norway,Oslo,EMEA,NOR,Oracle,1072,Tech,SMB,Marketing Suite - Gold,3PHPTG5Z8H,14.900,5,0.0,1.0430
8245,8246,EMEA-2020-143581,2020-08-19,20200819,Kevin Wilkins,Finland,Espoo,EMEA,NOR,Lowes,1060,Retail,Strategic,Marketing Suite - Gold,5YX5FT352R,344.910,3,0.0,10.3473
5745,5746,AMER-2023-147886,2023-03-29,20230329,Gavin Clark,United States,Columbia,AMER,NAMER,Nestle,1089,Consumer Products,Strategic,SaaS Connector Pack,ZYCM6TY01T,21.560,7,0.0,6.8992
6763,6764,APJ-2022-144764,2022-09-03,20220903,Irene Wright,Australia,Sydney,APJ,ANZ,Chevron,1017,Energy,SMB,Marketing Suite - Gold,IXMT8NX32U,35.168,2,0.2,-8.3524


In [571]:
print("---SUMMARY STATISTICS---")
df[["sales", "quantity", "discount", "profit"]].describe()

---SUMMARY STATISTICS---


,sales,quantity,discount,profit
count,9994.000000,9994.000000,9994.000000,9994.000000
mean,229.858001,3.789574,0.156203,28.656896
std,623.245101,2.225110,0.206452,234.260108
min,0.444000,1.000000,0.000000,-6599.978000
25%,17.280000,2.000000,0.000000,1.728750
50%,54.490000,3.000000,0.200000,8.666500
75%,209.940000,5.000000,0.200000,29.364000
max,22638.480000,14.000000,0.800000,8399.976000


In [572]:
print("---SUMMARY OF CATEGORIAL VARIABLES---")
df.describe(include="str")

---SUMMARY OF CATEGORIAL VARIABLES---


,order_id,contact_name,country,city,region,subregion,customer,industry,segment,product,license
count,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994
unique,5009,793,48,262,3,12,99,10,3,14,9994
top,EMEA-2023-100111,Leonard Kelly,United States,London,EMEA,NAMER,Allianz,Finance,SMB,ContactMatcher,16GRM07R1K
freq,14,37,2001,922,4219,2507,192,2127,5191,1842,1


In [573]:
print(f"Number of unique customers: {len(df["customer"].unique())}")
print(f"Number of unique customer IDs: {len(df["customer_id"].unique())}")

Number of unique customers: 99
Number of unique customer IDs: 99


In [574]:
print("---SALES BY CATEGORIES---\n")
features = ["segment", "industry", "region",
            "subregion", "product", "country", 
            "city", "customer"]
features_top20 = ["country", "city", "customer"]

for feature in features:
    print(f"{feature}:\n")
    segment_profile = df.groupby([feature]).agg({
        "sales": ["count", "mean", "sum"],
        "profit": ["mean", "sum"],
        "discount": "mean"
    })
    segment_profile = segment_profile.sort_values(
    by=("sales", "count"), ascending=False
    ).reset_index()

    if feature in features_top20:
        print(f"{segment_profile.head(20).round(2)}\n")
    else:
        print(f"{segment_profile.round(2)}\n")

---SALES BY CATEGORIES---

segment:

      segment sales                     profit            discount
              count    mean         sum   mean        sum     mean
0         SMB  5191  223.73  1161401.34  25.84  134119.21     0.16
1   Strategic  3020  233.82   706146.37  30.46   91979.13     0.16
2  Enterprise  1783  240.97   429653.15  33.82   60298.68     0.15

industry:

            industry sales                    profit           discount
                     count    mean        sum   mean       sum     mean
0            Finance  2127  222.92  474150.48  23.67  50348.97     0.15
1             Energy  1316  231.49  304644.14  34.41  45282.31     0.16
2               Tech  1236  212.92  263169.03  19.92  24615.04     0.16
3      Manufacturing  1222  241.56  295192.38  31.43  38413.11     0.16
4         Healthcare  1049  260.42  273183.29  30.48  31969.09     0.16
5  Consumer Products  1021  219.44  224044.14  36.26  37018.01     0.16
6             Retail   972  229.50  2230

- Segment column looks suspicious. Majority of purchases were made by "small and medium business" segment, but we only have 99 customers and they all seem to be of the larger segment. We will disregard this column in later analysis. 

- Let's see whether all of the unprofitable countries have a discount level of 28% or larger

In [575]:
segment_profile = df.groupby(["country"]).agg({
    "sales": ["count", "mean", "sum"],
    "profit": ["mean", "sum"],
    "discount": "mean"
})

segment_profile.sort_values(by=("discount", "mean"), ascending=False).round(2)

sales                      profit           discount
                     count     mean        sum    mean       sum     mean
country                                                                  
Australia              492   162.94   80166.10  -25.63 -12607.89     0.39
Japan                  985   172.78  170188.05  -26.12 -25729.36     0.37
France                 587   198.49  116511.91  -26.51 -15559.96     0.33
Mexico                 469   166.86   78258.14  -36.19 -16971.38     0.32
Russia                 182   176.42   32108.12  -35.87  -6527.86     0.32
Chile                  224   157.51   35282.00  -15.30  -3427.92     0.30
Germany                383   233.61   89473.71   -8.88  -3399.30     0.30
South Africa           183   167.55   30661.87  -29.19  -5341.69     0.29
Belgium                124   140.57   17431.15   -9.60  -1190.47     0.29
Sweden                 249   223.31   55603.16  -30.08  -7490.91     0.28
Slovenia                 1  1603.14    1603.14  100.20    100.20     0.20
Taiwan                  21   208.69    4382.49   39.37    826.72     0.09
Iceland                  4   302.46    1209.82   46.48    185.92     0.08
United States         2001   228.73  457687.63   38.17  76381.39     0.07
Canada                 506   273.99  138641.27   66.01  33402.65     0.06
Israel                  39   428.95   16729.10   85.05   3316.77     0.06
Turkey                  53   211.70   11220.06   48.05   2546.53     0.06
New Zealand             37   129.28    4783.52   31.27   1157.12     0.06
United Kingdom        1141   274.47  313169.88   65.72  74989.09     0.05
Netherlands             45   225.98   10169.11   67.05   3017.14     0.02
Norway                  56   404.07   22627.96  130.10   7285.63     0.02
South Korea            135   212.11   28634.43   50.26   6785.50     0.02
Greece                  27   270.09    7292.52   63.20   1706.50     0.01
Saudi Arabia            82   163.22   13384.36   42.82   3511.49     0.01
Brazil                 255   299.10   76269.61   95.93  24463.19     0.01
Philippines             96   285.95   27451.07  103.93   9977.37     0.01
China                  105   225.77   23705.52   66.96   7031.18     0.01
Argentina              130   275.11   35764.31   75.18   9772.91     0.00
Costa Rica              38   196.45    7464.93   53.61   2037.09     0.00
Croatia                 12   109.63    1315.56   32.90    394.83     0.00
Colombia                42   201.95    8481.71   42.12   1769.06     0.00
Austria                 10   286.50    2865.02  105.96   1059.59     0.00
Finland                184   266.83   49095.84   88.32  16250.04     0.00
Egypt                   11   811.76    8929.37  204.09   2244.98     0.00
Denmark                  8   158.82    1270.53   56.81    454.49     0.00
Czech Republic          42   219.45    9217.03   52.29   2196.10     0.00
India                  149   359.43   53555.36  123.38  18382.94     0.00
Indonesia               24   121.43    2914.31   34.85    836.44     0.00
Ireland                126   272.21   34298.14   81.34  10249.16     0.00
Italy                  110   291.95   32114.61   76.38   8401.80     0.00
Poland                  66   336.44   22205.15   97.52   6436.21     0.00
Luxembourg              89   335.54   29863.15  121.61  10823.19     0.00
Qatar                    7   131.42     919.91   32.88    230.15     0.00
Portugal                61   319.85   19510.64   94.87   5786.83     0.00
Singapore               66   298.23   19683.39   73.54   4853.96     0.00
Spain                  224   315.34   70636.72   83.03  18597.95     0.00
United Arab Emirates    60   194.64   11678.13   66.81   4008.69     0.00
Ukraine                 53   203.23   10771.34   59.87   3172.98     0.00

Let's see the profit-discount patterns of most and least profitable companies

In [576]:
segment_profile = df.groupby(["customer"]).agg({
    "sales": ["count", "mean", "sum"],
    "profit": ["mean", "sum"],
    "discount": "mean"
})

segment_profile.sort_values(by=("profit", "sum"), ascending=False).round(2).head(10)

sales                    profit           discount
                       count    mean       sum    mean       sum     mean
customer                                                                 
Valero Energy            105  392.58  41220.42   98.18  10308.63     0.15
Coca-Cola                 81  353.63  28643.80  116.65   9449.02     0.17
Trafigura Group          103  324.73  33447.13   86.10   8867.83     0.07
Mondelez International   143  230.72  32993.05   59.98   8577.65     0.13
Lowes                    110  366.91  40360.16   72.16   7937.49     0.18
Lukoil                   116  303.44  35199.18   61.35   7117.09     0.15
Siemens                  170  211.81  36008.37   38.35   6519.51     0.18
Bank of America Corp.    132  312.55  41255.95   48.86   6449.86     0.13
Kroger                   135  220.62  29783.46   46.34   6256.10     0.14
Anthem                   134  415.82  55719.21   44.43   5953.20     0.12

In [583]:
segment_profile.sort_values(by=("profit", "sum"), ascending=True).round(2).head(10)

sales                   profit          discount
                          count    mean       sum   mean      sum     mean
customer                                                                  
Allstate                    105  380.40  39941.64 -26.63 -2796.29     0.12
Bosch                       119  213.82  25445.00 -15.33 -1823.78     0.16
Nissan Motor                 70  323.58  22650.82 -22.65 -1585.19     0.18
Costco Wholesale             62  351.56  21796.70 -21.85 -1354.85     0.14
Walgreens                    68  212.07  14420.62 -10.89  -740.67     0.15
Sprint Nextel               101  186.65  18852.08  -4.58  -462.66     0.17
Morgan Stanley              126  227.23  28631.45  -2.87  -361.81     0.17
HonHai Precision Industry    88  209.74  18457.42  -1.55  -136.70     0.13
HSBC Holdings                83  273.54  22703.84  -1.26  -104.40     0.19
Gazprom                      53  101.94   5402.90  -0.26   -13.97     0.18

### 3. FEATURE ENGINEERING

- Because the business is supposed to be B2B SaaS, we need to treat all the purchases as subscriptions.
    - We assume than each subscription lasts 12 months, with payments occuring monthly.
    - Because the data is generated randomly, I expect customer to be purchasing at random, with some churning and reactivating and with others purchasing a subscription before expiration date.
    - Nevertheless, this new feature will make the analysis appear more realistic.

- New columns: 1) subscription end date; 2) monthly payment size; 3) monthly profit size

In [578]:
df["subscription_end"] = df["order_date"] + pd.DateOffset(years=1) - pd.Timedelta(days=1)
df['monthly_payment'] = df['sales'] / 12
df['monthly_profit'] = df['profit'] / 12

### 4. INITIAL HYPOTHESIS TESTING

- H1. Effect of discount on profitability.
    - Profit loss is closely related to the size of the discount. 
    - Average discount for unprofitable deals should be significantly larger than for profitable deals.
- H2. Effect of discount on sales seasonality.
    - There will be a visible seasonal sales trend that is driven by the size of the discount.

In [579]:
print("---H1. EFFECT OF DISCOUNT ON PROFITABILITY---\n")
unprofitable_deals = df[df["profit"] <= 0]
profitable_deals = df[df["profit"] > 0]
print(f"Average discount on unprofitable deals: {unprofitable_deals["discount"].mean():.1%}")
print(f"Average discount on profitable deals: {profitable_deals["discount"].mean():.1%}\n")
print(f"Total count of unprofitable deals: {len(unprofitable_deals)}")
print(f"Total count of profitable deals: {len(profitable_deals)}")

---H1. EFFECT OF DISCOUNT ON PROFITABILITY---

Average discount on unprofitable deals: 46.9%
Average discount on profitable deals: 8.1%

Total count of unprofitable deals: 1936
Total count of profitable deals: 8058


There is a 38.8% difference in averages, which supports Hypothesis 1.

In [580]:
print("---H2. SALES SEASONALITY---\n")
df["order_quarter"] = df["order_date"].dt.to_period("Q")
quarterly_trends = df.groupby("order_quarter").agg({"sales": ["count", "sum"], "profit": "sum", "discount": "mean"})
quarterly_trends.round(2)

---H2. SALES SEASONALITY---



sales               profit discount
              count        sum       sum     mean
order_quarter                                    
2020Q1          282   74447.80   3811.23     0.16
2020Q2          392   86538.76  11204.07     0.15
2020Q3          564  143633.21  12804.72     0.16
2020Q4          755  179627.73  21723.95     0.17
2021Q1          260   68851.74   9264.94     0.15
2021Q2          444   89124.19  12190.92     0.17
2021Q3          592  130259.58  16853.62     0.15
2021Q4          806  182297.01  23309.12     0.15
2022Q1          332   92596.42  11446.34     0.15
2022Q2          594  135370.11  16084.91     0.16
2022Q3          741  144614.43  16153.50     0.16
2022Q4          913  235892.87  38042.18     0.15
2023Q1          494  118895.62  21772.23     0.15
2023Q2          692  134023.41  17165.76     0.17
2023Q3          907  200433.17  26913.45     0.15
2023Q4         1226  280594.83  27656.08     0.16

In every single year of data, seasonal trend is the same, with Q4 having the largest sales and Q1 having the lowest. There is support for the Hypothesis 2.

### 5. Exporting the file to SQL

In [581]:
df.columns

Index(['transaction_id', 'order_id', 'order_date', 'date_key', 'contact_name',
       'country', 'city', 'region', 'subregion', 'customer', 'customer_id',
       'industry', 'segment', 'product', 'license', 'sales', 'quantity',
       'discount', 'profit', 'subscription_end', 'monthly_payment',
       'monthly_profit', 'order_quarter'],
      dtype='str')

In [582]:
columns_to_remove = [
    "date_key", "license", "customer_id", 
    "segment", "order_quarter"
]

df = df.drop(columns=columns_to_remove)
df.to_csv("saas_sales_cleaned.csv", index=False)

Removed redundant columns:

    - date_key (same as order_date), 
    - license (unique identifier, same as transaction_id), 
    - customer_id (same as customer),
    - segment (inaccurate feature)